# Explore whisper models

In [ ]:
import datetime
import pathlib

import isodate
from dotenv import load_dotenv


def get_isodatetime():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.datetime_isoformat(current)
    return iso_value

def get_isodate():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.date_isoformat(current)
    return iso_value

import audio_transcription
import summarize_transcript
import transcript_merger

#local functions
import video_extraction
import whisper

# Load environment variables
load_dotenv()
logger = video_extraction.logger

logger.info("Available Whisper transcription models:")
logger.info(whisper.available_models())

In [ ]:
mic_audio_filename = "microphone_audio.wav"
desktop_audio_filename = "desktop_audio.wav"
output_dir = "./2024-07-12-output"
whisper_models = ['tiny','tiny.en','base','base.en','medium','large']

#setup folder
output_path = pathlib.Path(output_dir)
if not output_path.exists():
    output_path.mkdir()
print(f"Output dir is: {output_path}")

# -- Transcribe audio
output_path = pathlib.Path(output_dir)
mic_audio_path = output_path / mic_audio_filename
desktop_audio_path = output_path / desktop_audio_filename

for whisper_model in whisper_models:
    print(f"Transcribing with: {whisper_model}")
    try:
        audio_transcription.process_audio_files(
            str(mic_audio_path),
            str(desktop_audio_path),
            output_dir=str(output_path),
            whisper_model_name=whisper_model
        )
    except Exception as e:
        logger.error(f"An error occurred during the transcription process: {str(e)}")
        logger.error("\n\n Continuing to next model")

# Evals

## Setup Eval info

In [ ]:
def add_transcript(transcript_info:list, model:str, microphone_filename:str, desktop_filename:str)->list:
    """Add a new transcription to the list to eval"""
    info = {
        'model' : model,
        'microphone' : microphone_filename,
        'desktop' : desktop_filename
    }
    transcript_info.append(info)
    return info

transcript_info = []
whisper_models = ['tiny','tiny.en','base','base.en','medium','large']
for model in whisper_models:
    logger.info(f"------------------ {model} -----------------------------")
    logger.info(f"Adding eval for '{model}'")
    microphone_filename = f"microphone-whisper-{model}_transcription.txt"
    desktop_filename = f"desktop-whisper-{model}_transcription.txt"
    info = add_transcript(transcript_info, model, microphone_filename, desktop_filename)
    _ = [logger.info(f"  {k} : {v}") for k,v in info.items()]
    logger.info("\n")

logger.info("############## Completed with no errors ####################")


## Merge transcripts

In [ ]:
def add_merged_transcript(info:dict, merged_filename:str)->list:
    """Add a merged transcription to the list to eval"""
    info.update({'merged' : merged_filename,})
    return info




output_dir = "./2024-07-12-output"

for info in transcript_info:
    logger.info(f"------------------ {info['model']} -----------------------------")
    mic_transcript_path = pathlib.Path(output_dir) / info['microphone']
    desktop_transcript_path = pathlib.Path(output_dir) / info['desktop']
    transcript_file_path = pathlib.Path(output_dir) / f"merged-whisper-{info['model']}-transcript.json"
    logger.info("Merging Transcription to:")
    logger.info(f"- {transcript_file_path}")
    try:
        transcript_json = transcript_merger.process_transcripts(
            mic_transcript_file=str(mic_transcript_path),
            desktop_transcript_file=str(desktop_transcript_path),
            output_file=str(transcript_file_path),
            parse_overlaps=False
        )
        info = add_merged_transcript(info, str(transcript_file_path))
    except Exception as e:
        print("An error occurred during the merging process:")
        print(f"{str(e)}")
    logger.info("\n")

logger.info("############## Completed with no errors ####################")

In [ ]:
def export_to_markdown(content, filename):
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(content)
    return

output_dir = "./2024-07-12-output"
summarize_model_name = 'claude-3-5-sonnet'
summarize = True

for info in transcript_info:
    logger.info(f"------------------ {info['model']} -----------------------------")
    transcript_file_path = pathlib.Path(info['merged'])
    summary_file_path = pathlib.Path(output_dir) / f"{info['model']}-summary.txt"
    summary_markdown_path = str(summary_file_path.with_suffix('.md'))
    logger.info(f"Summary filename: {summary_file_path}")
    logger.info(f"Summary markdown: {summary_markdown_path}")

    # Estimate tokens and price
    transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
    transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
    total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
        transcript_text,
        model=summarize_model_name
    )

    if summarize:
        logger.info("Summarizing")
        try:
            summary = summarize_transcript.generate_summary(transcript_text, model_name=summarize_model_name)
            summarize_transcript.save_summary(
                summary,
                output_file=str(summary_file_path)
            )
            logger.info(f"Summary generated and saved to {summary_file_path}")
            export_to_markdown(summary, filename=summary_markdown_path)
        except Exception as e:
            logger.error(f"An error occurred during summarization: {str(e)}")
    else:
        logger.info("Skipping summarize step")

    logger.info("\n")

logger.info("############## Completed with no errors ####################")